# Adversarial Reasoning Gym — One-notebook pipeline

End-to-end Colab notebook for the OpenEnv code-debugging environment. Runs the complete flow:

**Free T4** sections:
1. Setup
2. Connect to env (smoke test)
3. Baseline measurement (Qwen3-0.6B on 50 scenarios)
4. *Optional* SFT format warmup
5. Sanity-check GRPO training (30 iters, 0.6B)

**A100** sections (~$25–28):

6. Real GRPO training (300 iters, Qwen3-1.7B, staged curriculum)

**Free T4 again** for evaluation:

7. Compare base vs trained on identical seeds
8. Pick demo episodes + render plots
9. Push to HuggingFace Hub & Spaces

Each section is independent — re-run only the parts you need. Section 6 is the only paid step; everything else is free.

---
## §1. Setup

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q unsloth==2025.4.0 trl==0.29.0 transformers>=4.45.0 accelerate bitsandbytes datasets peft
!pip install -q fastapi pydantic httpx matplotlib

In [ ]:
import os
REPO_URL = 'https://github.com/<your-org>/adversarial-reasoning-gym.git'  # <-- edit me
if not os.path.exists('adversarial-reasoning-gym'):
    !git clone $REPO_URL
%cd adversarial-reasoning-gym

---
## §2. Connect to env (smoke test)

Loads the env directly in-process. Confirms templates and sandbox are working before we touch any model.

In [ ]:
from server.environment import AdversarialReasoningEnv
from server.models import Action

env = AdversarialReasoningEnv(seed=0)
obs = env.reset(difficulty='easy')
print('Function:', env._scenario.function_name)
print('User points at line', env._scenario.wrong_line, '— actual bug on line', env._scenario.bug.bug_line)
print()
print(obs.task_description[:300])

# One-shot oracle fix to confirm the reward path lights up.
for tool, args in [('read_code', {}), ('run_tests', {}),
                   ('apply_fix', {'line_number': env._scenario.bug.bug_line,
                                  'new_code': env._scenario.bug.original_code})]:
    env.step(Action(action_type='tool_call', tool_name=tool, tool_args=args))
final = env.step(Action(action_type='submit'))
print(f'\nOracle fix reward: {final.reward:.3f}, all_pass={final.info["all_pass"]}')

---
## §3. Phase 2 — Baseline measurement (free T4)

Run **untrained** Qwen3-0.6B on 50 scenarios. Capture the four metrics: accuracy, cave rate, investigation depth, resistance rate. These are your "before" numbers for the final report.

Includes a format smoke test: if the model can't produce parseable actions, jump to §4 for an SFT warmup before continuing.

In [ ]:
BASE_MODEL = 'Qwen/Qwen3-0.6B-Instruct'
EVAL_SEED = 42
EVAL_N = 50

In [ ]:
# Format smoke test — DO NOT skip.
from eval_llm import _load_model, _make_llm_policy
from server.rollout import build_prompt
from server.action_parser import parse_action

model, tokenizer = _load_model(BASE_MODEL, load_in_4bit=False)
policy = _make_llm_policy(model, tokenizer, max_new_tokens=256, temperature=0.0)

test_obs = AdversarialReasoningEnv(seed=99).reset(difficulty='easy')
test_prompt = build_prompt(history=[], current_obs=test_obs)
completion = policy(test_prompt)
print('=== Sample completion ===')
print(completion[:500])
action, warnings = parse_action(completion)
print(f'\nParsed: {action.action_type}/{action.tool_name}  args={action.tool_args}')
print(f'Warnings: {warnings or "none"}')

In [ ]:
# Free model from memory before the real eval (eval_llm reloads it).
del model, tokenizer, policy
import gc, torch; gc.collect(); torch.cuda.empty_cache()

!python eval_llm.py \
    --model $BASE_MODEL \
    --n $EVAL_N --seed $EVAL_SEED --difficulty easy \
    --temperature 0.0 \
    --out baseline.json --log baseline_episodes.jsonl

In [ ]:
import json
print(json.dumps(json.load(open('baseline.json')), indent=2))

**Decision rule.** If `accuracy > 5%` and `cave_rate > 30%`, skip §4 and continue. If `accuracy ~ 0` AND `cave_rate ~ 0` (everything fails AND nothing caves) → format collapse → run §4 next.

---
## §4. *Optional* SFT format warmup

Skip if the §3 smoke test showed `Warnings: none`. Otherwise:

1. Generate 400 episodes worth of `(prompt, completion)` pairs using the heuristic policy (perfect format, oracle fixes).
2. Run a 2-epoch SFT on Qwen3-0.6B.
3. Re-measure baseline (saved as `baseline_sft.json`).

In [ ]:
!python sft_warmup.py --generate-only --n 400 --out sft_data.jsonl --seed 0

import json
lines = open('sft_data.jsonl').readlines()
print(f'Generated {len(lines)} (prompt, completion) pairs')

In [ ]:
!python sft_warmup.py \
    --model $BASE_MODEL --out sft_data.jsonl \
    --output-dir ckpts/sft-warmup \
    --epochs 2 --batch-size 4 --grad-accum 2 --lr 2e-5

In [ ]:
# Re-measure baseline on the SFT'd model. This becomes the starting point
# for §5 / §6 training; the original baseline.json stays as your pure 'before'.
!python eval_llm.py \
    --model ./ckpts/sft-warmup \
    --n $EVAL_N --seed $EVAL_SEED --difficulty easy \
    --temperature 0.0 \
    --out baseline_sft.json --log baseline_sft_episodes.jsonl

If you ran §4, set `START_FROM` below to `'./ckpts/sft-warmup'`. Otherwise leave it as the raw base model.

In [ ]:
import os
START_FROM = './ckpts/sft-warmup' if os.path.exists('ckpts/sft-warmup') else BASE_MODEL
print('Will train from:', START_FROM)

---
## §5. Phase 3 — Sanity-check GRPO (free T4, 0.6B)

30 outer iterations × 4 rollouts on the 0.6B model, easy difficulty only.
**Goal:** verify the loop actually trains. If reward trends up, proceed to §6 (paid). If flat or collapsing, do not pay for A100 — fix rewards instead.

~30–60 min on T4.

In [ ]:
!python train.py \
    --model $START_FROM \
    --outer-iterations 30 \
    --rollouts-per-iter 4 \
    --num-generations 4 \
    --max-new-tokens 256 \
    --max-seq-length 1536 \
    --lr 1e-5 --grad-accum 4 --gamma 0.95 \
    --curriculum easy \
    --output-dir ckpts/phase3-sanity \
    --output-log phase3_train.jsonl \
    --episodes-log phase3_episodes.jsonl --seed 0

In [ ]:
# Reward trend check — the gating decision for whether to spend on §6.
import json, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open('phase3_train.jsonl')]
rewards = [r['reward'] for r in rows]
def rolling(xs, w=10):
    return [sum(xs[max(0,i-w+1):i+1]) / min(w, i+1) for i in range(len(xs))]
plt.figure(figsize=(9,3.5))
plt.plot(rewards, alpha=0.4); plt.plot(rolling(rewards), linewidth=2.5)
plt.axhline(0, color='gray', linewidth=0.5); plt.grid(alpha=0.3)
plt.title(f'Phase 3 reward — should trend up ({len(rows)} episodes)'); plt.show()

f10 = sum(rewards[:10]) / max(1, len(rewards[:10]))
l10 = sum(rewards[-10:]) / max(1, len(rewards[-10:]))
delta = l10 - f10
print(f'mean(first10)={f10:.3f}  mean(last10)={l10:.3f}  delta={delta:+.3f}')
print('[OK] proceed to §6.' if delta > 0.05 else
      '[WARN] flat — consider 20 more iters.' if delta > -0.02 else
      '[STOP] reward collapsing — fix rewards before paying.')

---
## §6. Phase 4 — Real GRPO training (A100, $25–28)

**Switch runtime to A100 before running this section.** Runtime → Change runtime type → A100.

300 outer iterations × 8 rollouts × 8 GRPO completions on Qwen3-1.7B. Staged curriculum: easy 1–50, medium 50–150, adaptive 150+. ~3 hours.

**Watch §6.5 (live monitoring) every 20 steps.** If reward stalls for 30+ iters, kill and inspect the episodes log.

In [ ]:
from huggingface_hub import login
login()  # paste a write-scope token from https://huggingface.co/settings/tokens

In [ ]:
PROD_MODEL = 'Qwen/Qwen3-1.7B-Instruct'  # NOT the SFT'd 0.6B — Phase 4 uses the bigger model.
HUB_REPO = 'your-username/qwen3-1.7b-arg'  # <-- edit me

In [ ]:
!python train.py \
    --model $PROD_MODEL \
    --outer-iterations 300 \
    --rollouts-per-iter 8 \
    --num-generations 8 \
    --max-new-tokens 256 --max-seq-length 2048 \
    --lr 1e-5 --grad-accum 4 --gamma 0.95 \
    --curriculum phase4 \
    --save-every 20 --hub-repo $HUB_REPO \
    --output-dir ckpts/phase4-real \
    --output-log phase4_train.jsonl \
    --episodes-log phase4_episodes.jsonl \
    --seed 0

### §6.5 Live monitoring

Run this cell every 20 minutes while §6 is training. Reward + cave-rate + last episode trace.

In [ ]:
import json, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open('phase4_train.jsonl') if l.strip()]
if rows:
    rewards = [r['reward'] for r in rows]; caved = [1 if r['caved'] else 0 for r in rows]
    def rolling(xs, w=20):
        return [sum(xs[max(0,i-w+1):i+1]) / min(w, i+1) for i in range(len(xs))]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(rewards, alpha=0.4); axes[0].plot(rolling(rewards), linewidth=2.5)
    axes[0].axhline(0, color='gray', linewidth=0.5)
    axes[0].set_title(f'Reward ({len(rows)} eps)'); axes[0].grid(alpha=0.3)
    axes[1].plot(rolling(caved), linewidth=2.5, color='tab:red')
    axes[1].set_title('Cave rate (rolling 20)'); axes[1].set_ylim(0, 1); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

import os
if os.path.exists('phase4_episodes.jsonl'):
    last = json.loads(open('phase4_episodes.jsonl').readlines()[-1])
    print(f'\n--- latest: iter {last["iter"]} fn={last["function"]} pass={last["all_pass"]} caved={last["caved"]} reward={last["total_reward"]:.2f} ---')
    for i, s in enumerate(last['steps'][:6]):
        print(f'  {i}: {s["action_type"]}/{s["tool_name"]} {s["tool_args"]} reward={s["reward"]:.2f}')

---
## §7. Phase 5 — Compare base vs trained

**Switch back to T4** to save credits. Run base + trained on the same 100 seeds and capture the headline numbers.

In [ ]:
PROD_BASE = 'Qwen/Qwen3-1.7B-Instruct'
TRAINED = HUB_REPO  # the model you pushed in §6
EVAL_SEED = 42
EVAL_N = 100

In [ ]:
!python eval_llm.py --model $PROD_BASE --n $EVAL_N --seed $EVAL_SEED --difficulty easy \
    --temperature 0.0 --out base_results.json --log base_episodes.jsonl

In [ ]:
!python eval_llm.py --model $TRAINED --n $EVAL_N --seed $EVAL_SEED --difficulty easy \
    --temperature 0.0 --out trained_results.json --log trained_episodes.jsonl

In [ ]:
import json
b = json.load(open('base_results.json'))
t = json.load(open('trained_results.json'))
print(f'{"":25s} {"BASE":>10s}      {"TRAINED":>10s}')
for k in ('accuracy','cave_rate','resistance_rate'):
    print(f'{k:25s} {b[k]:>10.2%}  ->  {t[k]:>10.2%}')
print(f'{"avg_investigation_depth":25s} {b["avg_investigation_depth"]:>10.2f}  ->  {t["avg_investigation_depth"]:>10.2f}')

---
## §8. Plots + demo picks

In [ ]:
import json
json.dump({
    'n': b['n'],
    'base':    {k: b[k] for k in ('accuracy','cave_rate','avg_investigation_depth','resistance_rate','n')},
    'trained': {k: t[k] for k in ('accuracy','cave_rate','avg_investigation_depth','resistance_rate','n')},
}, open('results.json', 'w'), indent=2)

import os
training_log = 'phase4_train.jsonl' if os.path.exists('phase4_train.jsonl') else 'phase3_train.jsonl'
!python plot_results.py --training-log $training_log --results results.json --out-dir plots

In [ ]:
from IPython.display import Image, display
import os
for name in ['1_reward.png','2_fix_accuracy.png','3_cave_rate.png',
             '4_investigation_depth.png','5_base_vs_trained.png','6_difficulty_axes.png']:
    p = f'plots/{name}'
    if os.path.exists(p):
        print(name); display(Image(p))

In [ ]:
!python pick_demo.py --base base_episodes.jsonl --trained trained_episodes.jsonl --top 5
print(open('demo_picks.md').read())

In [ ]:
# Print one head-to-head trace for the blog/video.
DEMO_EP = 0   # change to an interesting ep from demo_picks.md
import json
base_eps = {b['ep']: b for b in [json.loads(l) for l in open('base_episodes.jsonl')]}
trained_eps = {t['ep']: t for t in [json.loads(l) for l in open('trained_episodes.jsonl')]}
print('=== BASE ===');    print(json.dumps(base_eps[DEMO_EP], indent=2))
print('\n=== TRAINED ==='); print(json.dumps(trained_eps[DEMO_EP], indent=2))

---
## §9. Phase 6 — Push to Hugging Face Hub & Spaces

Final packaging. Trained model goes to a model repo; the env itself goes to a Space. See [`PHASE6_SUBMISSION.md`](PHASE6_SUBMISSION.md) for the full checklist (README polish, blog draft, video script).

In [ ]:
# Confirm the trained model is on the Hub (was auto-pushed by §6 if --hub-repo was set).
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files(HUB_REPO)
print(f'Files in {HUB_REPO}: {len(files)}')
for f in files[:20]: print(' ', f)

In [ ]:
# Optional: download artifacts so they survive Colab session end.
from google.colab import files
for name in ['baseline.json', 'baseline_sft.json',
             'base_results.json', 'trained_results.json',
             'phase3_train.jsonl', 'phase4_train.jsonl',
             'demo_picks.md', 'demo_picks.json']:
    if os.path.exists(name):
        files.download(name)

---
**Done.** Headline numbers are in `results.json`, plots in `plots/`, demo episodes in `demo_picks.md`. For HF Space deployment, blog post, and submission checklist: [`PHASE6_SUBMISSION.md`](PHASE6_SUBMISSION.md).